In [2]:
from controllables.energyplus import System

system = System(
            building='all_room_have_hvac_t_22.idf',
            weather='SGP_SG_Singapore-Paya.Lebar.AB.486940_TMYx.2009-2023.epw',
            # TODO
            report='tmp/',
            repeat=True,
        )
        #system.add('logging:progress')

system.add('logging:progress').start()

/home/AD/user/lab/hvacmarl6e43/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  0%|          | 0/100.0 [00:00<?, ?it/s]

System({'building': 'all_room_have_hvac_t_22.idf', 'weather': 'SGP_SG_Singapore-Paya.Lebar.AB.486940_TMYx.2009-2023.epw', 'report': 'tmp/', 'repeat': True})

  0%|          | 0/100.0 [00:00<?, ?it/s, EnergyPlus, Version 23.2.0-7636e6b3e9, YMD=2025.08.05 02:20]

  0%|          | 0.0/100.0 [00:02<?, ?it/s, Warming up {2}]                                                      

In [ ]:
system.variables.available_keys()

  6%|▌         | 6.0/100.0 [00:03<00:30,  3.04it/s, Starting Simulation at 07/01/2002 for SITE (01-07:01-08)]

Loading ITables v2.4.4 from the internet... (need help?)
Loading ITables v2.4.4 from the internet... (need help?)


100%|██████████| 100.0/100.0 [00:08<00:00, 24.92it/s, EnergyPlus, Version 23.2.0-7636e6b3e9, YMD=2025.08.05 02:20]          

In [3]:
from controllables.core import TemporaryUnavailableError
from controllables.energyplus import (
    Actuator, 
    OutputVariable,
    OutputMeter
)
import pythermalcomfort as pytc
import numpy as _numpy_
from controllables.core import Variable, BaseVariable
import itables as _itables_

 24%|██▍       | 24.0/100.0 [00:05<00:06, 12.19it/s, Starting Simulation at 07/01/2002 for SITE (01-07:01-08)]   

In [4]:
class PMVVariable(BaseVariable):
        def __init__(
            self, 
            tdb: BaseVariable,
            tr: BaseVariable,
            rh: BaseVariable,
            metab_rate=1.5, clothing=.5, pmv_limit=.5,
        ):
            self.tdb = tdb
            self.tr = tr
            self.rh = rh
            self._metab_rate = _numpy_.asarray(metab_rate)
            self._clothing = _numpy_.asarray(clothing)
            self._pmv_limit = _numpy_.asarray(pmv_limit)
        
        @property
        def value(self):
            res = pytc.models.pmv_ppd(
                tdb=self.tdb.value, 
                tr=self.tr.value, 
                # calculate relative air speed
                vr=pytc.utilities.v_relative(v=0.1, met=self._metab_rate), 
                rh=self.rh.value, 
                met=self._metab_rate, 
                # calculate dynamic clothing
                clo=pytc.utilities.clo_dynamic(clo=self._clothing, met=self._metab_rate),
                limit_inputs=False ,
            )['pmv']

            return res


In [5]:
from controllables.core.tools.records import VariableRecords
tdb = system[OutputVariable.Ref('Zone Mean Air Temperature','1FFIRSTFLOORWEST:OPENOFFICE')]
tr = system[OutputVariable.Ref('Zone Mean Radiant Temperature','1FFIRSTFLOORWEST:OPENOFFICE')]
rh = system[OutputVariable.Ref('Zone Air Relative Humidity','1FFIRSTFLOORWEST:OPENOFFICE')]

pmv= PMVVariable(tdb, tr, rh)
records = VariableRecords({
    '🕰️': system['wallclock:calendar'],
    # 'Electricity:Facility': 
    #     system[OutputMeter.Ref('Electricity:Facility')],
    'elec':system[OutputMeter.Ref('Electricity:HVAC')],
    'temp':system[OutputVariable.Ref('Zone Mean Air Temperature','1FFIRSTFLOORWEST:OPENOFFICE')],
    'pmv':pmv,
    # 'pmv_system':system[OutputVariable.Ref('Zone Thermal Comfort Fanger Model PMV','PEOPLE 1FFIRSTFLOORWEST:OPENOFFICE')],
    'occupancy':system[OutputVariable.Ref('Schedule Value','Office_OpenOff_Occ')]
},maxlen = 10000).watch(system.events['timestep'])

 26%|██▌       | 26.0/100.0 [00:05<00:05, 12.38it/s, Starting Simulation at 07/01/2002 for SITE (01-07:01-08)]/home/AD/user/lab/hvacmarl6e43/.venv/lib/python3.11/site-packages/controllables/energyplus/variables.py:396: RuntimeWarning: OutputVariable(OutputVariable.Ref(type='Zone Mean Air Temperature', key='1FFIRSTFLOORWEST:OPENOFFICE')) requested while Kernel() is running; It may not be available until the next run. More info: https://energyplus.readthedocs.io/en/latest/datatransfer.html#datatransfer.DataExchange.request_variable
  _warnings_.warn(
/home/AD/user/lab/hvacmarl6e43/.venv/lib/python3.11/site-packages/controllables/energyplus/variables.py:396: RuntimeWarning: OutputVariable(OutputVariable.Ref(type='Zone Mean Radiant Temperature', key='1FFIRSTFLOORWEST:OPENOFFICE')) requested while Kernel() is running; It may not be available until the next run. More info: https://energyplus.readthedocs.io/en/latest/datatransfer.html#datatransfer.DataExchange.request_variable
  _warnings_.wa

In [6]:
df = records.dataframe()
df.to_csv('tmp/records_baseline_t_22.csv', index=False)
_itables_.show(df)

Loading ITables v2.4.4 from the internet... (need help?)


In [7]:
df.describe()

,elec,temp,pmv
count,4.000000e+00,4.000000,4.00000
mean,1.043060e+07,20.610231,-0.66500
std,0.000000e+00,0.049347,0.01291
min,1.043060e+07,20.553473,-0.68000
25%,1.043060e+07,20.581287,-0.67250
50%,1.043060e+07,20.609651,-0.66500
75%,1.043060e+07,20.638594,-0.65750
max,1.043060e+07,20.668147,-0.65000


In [ ]:
import pandas
train = pandas.read_csv("tmp/records_single.csv")
train['🕰️'] = pandas.to_datetime(train['🕰️'])
_itables_.show(train)

 29%|██▉       | 28.999999999999996/100.0 [00:05<00:05, 12.18it/s, Starting Simulation at 07/01/2002 for SITE (01-07:01-08)]

FileNotFoundError: [Errno 2] No such file or directory: 'tmp/records_single.csv'

 85%|████████▌ | 85.0/100.0 [00:19<00:01, 12.89it/s, Continuing Simulation at 07/21/2002 for SITE (01-07:01-08)]            

In [ ]:
train.describe()

,🕰️,pmv,AHU COOLING COIL
count,36864,36864.000000,36864.000000
mean,2002-07-17 00:08:25.202966016,-0.361850,104020.242623
min,2002-01-01 00:00:00,-1.230000,61320.604462
25%,2002-07-09 00:10:00,-0.620000,87008.601593
50%,2002-07-17 00:10:00,-0.310000,94517.785299
75%,2002-07-25 00:10:00,-0.090000,121222.399098
max,2002-08-02 00:00:00,0.570000,174996.840643
std,NaN,0.315945,24294.228637


In [ ]:
import plotly

fig = plotly.graph_objs.Figure()
fig.add_scatter(x=df['🕰️'], y=df['pmv'], name='Baseline')
fig.add_scatter(x=train['🕰️'], y=train['pmv'], name='TODO')
fig

In [ ]:
import plotly.express as px

px.scatter(x=df['🕰️'], y=df['pmv'], title='Baseline')
px.scatter(x=train['🕰️'], y=train['pmv'], title='TODO')
